# **INTRODUCTION**

Retailers and FMCG companies increasingly rely on data-driven insights to understand customer purchasing behavior and optimize merchandising strategies. Market Basket Analysis is a widely used analytical technique that examines patterns of co-occurrence among products within customer transactions. By identifying which items are frequently purchased together, businesses can design effective product bundling, cross-selling, shelf placement, and promotional strategies.

In this notebook, association rule mining is applied to survey-based transaction data representing common Indian household purchases. Using the Apriori algorithm and association rules, the analysis uncovers frequent item combinations and directional relationships between products. The focus is not only on statistical significance but also on extracting actionable business insights relevant to retail decision-making in the Indian consumer context.

In [1]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [2]:
df = pd.read_excel('/content/Response Sheet.xlsx')
df.head()


,Bread,Milk,Curd,Atta,Rice,Cooking Oil,Sugar,Salt,Tea,Biscuits,Instant Noodles,Eggs,Onion,Potato,Tomato,Paneer,Pav,Shampoo,Toothpaste,Soap
0,1,1,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,0,1,1
1,1,0,1,1,1,0,0,1,1,1,1,1,1,1,1,0,1,1,1,1
2,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1
3,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,0,0,0,1,1
4,1,1,0,1,1,1,0,1,0,1,1,1,0,1,1,0,1,0,1,1


The dataset consists of binary survey responses where each row represents a customer transaction and each column represents a product category commonly purchased in Indian households. A value of 1 indicates that the product was purchased in the transaction, while 0 indicates it was not. This structure is appropriate for market basket analysis, as it captures co-purchase behavior across multiple items.

In [3]:
# Convert binary data to transactions
transactions = df.apply(
    lambda row: row.index[row == 1].tolist(),
    axis=1
).tolist()


In this step, the binary-encoded dataset is transformed into a list of transactions. For each customer, only the product names corresponding to a value of 1 are retained. This conversion is necessary because the Apriori algorithm operates on transaction-level item lists rather than raw binary matrices.

In [4]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)
df_encoded.head()


,Atta,Biscuits,Bread,Cooking Oil,Curd,Eggs,Instant Noodles,Milk,Onion,Paneer,Pav,Potato,Rice,Salt,Shampoo,Soap,Sugar,Tea,Tomato,Toothpaste
0,True,True,True,True,False,False,True,True,False,False,True,True,True,True,False,True,False,False,True,True
1,True,True,True,False,True,True,True,False,True,False,True,True,True,True,True,True,False,True,True,True
2,True,True,True,True,False,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True
3,True,True,False,False,False,False,True,False,False,False,False,True,True,False,False,True,False,False,False,True
4,True,True,True,True,False,True,True,True,False,False,True,True,True,True,False,True,False,False,True,True


The TransactionEncoder converts the list of transactions into a one-hot encoded DataFrame where each column represents a product and each row represents a transaction. The Boolean values indicate the presence or absence of items in a transaction. This format is required for efficient computation of frequent itemsets using the Apriori algorithm.

In [5]:
frequent_itemsets = apriori(
    df_encoded,
    min_support=0.2,   # adjust based on dataset size
    use_colnames=True
)

frequent_itemsets.sort_values(by='support', ascending=False)


,support,itemsets
0,1.000000,(Atta)
30,1.000000,"(Potato, Atta)"
15,1.000000,(Soap)
11,1.000000,(Potato)
115,1.000000,"(Soap, Instant Noodles)"
...,...,...
88548,0.222222,"(Toothpaste, Shampoo, Sugar, Rice, Eggs, Pav, ..."
88547,0.222222,"(Toothpaste, Sugar, Rice, Soap, Eggs, Pav, Bre..."
88546,0.222222,"(Toothpaste, Shampoo, Sugar, Rice, Eggs, Pav, ..."
16,0.222222,(Sugar)


The Apriori algorithm identifies frequently occurring item combinations based on a minimum support threshold of 20%. The results show a large number of frequent itemsets, including both single items and multi-item combinations. Items such as Atta, Potato, Soap, and their combinations appear with very high support, indicating that these are staple products commonly purchased together in the sample population.

In [7]:
fi_safe = frequent_itemsets[
    frequent_itemsets['itemsets'].apply(lambda x: len(x) in [1, 2, 3])
].copy()


To ensure computational stability and managerial interpretability, the frequent itemsets are filtered to include only itemsets of size 1, 2, and 3. High-order itemsets, although frequent, are often difficult to interpret and may lead to computational issues during rule generation. Retaining single-item itemsets is critical for accurate confidence and lift calculations.

In [8]:
fi_safe['itemsets'] = fi_safe['itemsets'].apply(frozenset)
fi_safe['support'] = fi_safe['support'].astype(float)


Itemsets are explicitly converted to frozenset format, and support values are cast to float type. This ensures compatibility with the association_rules function and prevents runtime errors arising from inconsistent data types. This step improves the robustness of the rule mining process.

In [9]:
rules = association_rules(
    fi_safe,
    metric="confidence",
    min_threshold=0.6
)

rules.head()


/usr/local/lib/python3.12/dist-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Atta),(Biscuits),1.000000,0.888889,0.888889,0.888889,1.0,1.0,0.0,1.0,0.0,0.888889,0.0,0.944444
1,(Biscuits),(Atta),0.888889,1.000000,0.888889,1.000000,1.0,1.0,0.0,inf,0.0,0.888889,0.0,0.944444
2,(Bread),(Atta),0.814815,1.000000,0.814815,1.000000,1.0,1.0,0.0,inf,0.0,0.814815,0.0,0.907407
3,(Atta),(Bread),1.000000,0.814815,0.814815,0.814815,1.0,1.0,0.0,1.0,0.0,0.814815,0.0,0.907407
4,(Cooking Oil),(Atta),0.555556,1.000000,0.555556,1.000000,1.0,1.0,0.0,inf,0.0,0.555556,0.0,0.777778


Association rules are generated using confidence as the primary metric with a minimum threshold of 60%. The results indicate strong directional relationships between products. For example, rules such as Atta → Bread and Biscuits → Atta exhibit high confidence, suggesting habitual and complementary purchasing patterns. Some rules show infinite conviction, indicating near-deterministic relationships within the dataset.

In [10]:
strong_rules = rules[
    (rules['lift'] > 1) &
    (rules['confidence'] >= 0.6)
]

strong_rules.sort_values(by='lift', ascending=False)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2935,(Paneer),"(Tea, Milk)",0.296296,0.296296,0.296296,1.000000,3.375000,1.0,0.208505,inf,1.000000,1.000000,1.000000,1.000000
2932,"(Tea, Milk)",(Paneer),0.296296,0.296296,0.296296,1.000000,3.375000,1.0,0.208505,inf,1.000000,1.000000,1.000000,1.000000
1893,(Sugar),"(Cooking Oil, Eggs)",0.222222,0.333333,0.222222,1.000000,3.000000,1.0,0.148148,inf,0.857143,0.666667,1.000000,0.833333
1892,"(Cooking Oil, Eggs)",(Sugar),0.333333,0.222222,0.222222,0.666667,3.000000,1.0,0.148148,2.333333,1.000000,0.666667,0.571429,0.833333
2239,"(Salt, Curd)",(Tea),0.259259,0.370370,0.259259,1.000000,2.700000,1.0,0.163237,inf,0.850000,0.700000,1.000000,0.850000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421,(Cooking Oil),"(Atta, Pav)",0.555556,0.814815,0.481481,0.866667,1.063636,1.0,0.028807,1.388889,0.134615,0.541667,0.280000,0.728788
1286,(Tea),"(Salt, Biscuits)",0.370370,0.666667,0.259259,0.700000,1.050000,1.0,0.012346,1.111111,0.075630,0.333333,0.100000,0.544444
3542,(Tea),"(Salt, Rice)",0.370370,0.666667,0.259259,0.700000,1.050000,1.0,0.012346,1.111111,0.075630,0.333333,0.100000,0.544444
3570,(Shampoo),"(Toothpaste, Rice)",0.555556,0.777778,0.444444,0.800000,1.028571,1.0,0.012346,1.111111,0.062500,0.500000,0.100000,0.685714


Rules are further filtered based on lift greater than 1 and confidence above 60%, ensuring that only statistically meaningful and non-random associations are retained. High-lift rules such as Paneer → Tea & Milk and Sugar → Cooking Oil & Eggs reveal strong complementary consumption behavior, which can be leveraged for product bundling and in-store promotions.

In [11]:
final_rules = strong_rules[[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]]

final_rules


,antecedents,consequents,support,confidence,lift
32,(Cooking Oil),(Biscuits),0.555556,1.000,1.125000
33,(Biscuits),(Cooking Oil),0.555556,0.625,1.125000
42,(Rice),(Biscuits),0.888889,1.000,1.125000
43,(Biscuits),(Rice),0.888889,1.000,1.125000
49,(Sugar),(Biscuits),0.222222,1.000,1.125000
...,...,...,...,...,...
3727,"(Sugar, Tomato)",(Toothpaste),0.222222,1.000,1.125000
3728,(Sugar),"(Toothpaste, Tomato)",0.222222,1.000,1.421053
3729,"(Tea, Toothpaste)",(Tomato),0.370370,1.000,1.227273
3730,"(Tea, Tomato)",(Toothpaste),0.370370,1.000,1.125000


Only the most relevant metrics—antecedents, consequents, support, confidence, and lift—are retained for final analysis. This simplifies interpretation and focuses attention on actionable insights rather than technical diagnostics, making the output suitable for managerial decision-making.

In [12]:
final_rules['antecedents'] = final_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
final_rules['consequents'] = final_rules['consequents'].apply(lambda x: ', '.join(list(x)))

final_rules


/tmp/ipython-input-2376321108.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_rules['antecedents'] = final_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
/tmp/ipython-input-2376321108.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_rules['consequents'] = final_rules['consequents'].apply(lambda x: ', '.join(list(x)))


,antecedents,consequents,support,confidence,lift
32,Cooking Oil,Biscuits,0.555556,1.000,1.125000
33,Biscuits,Cooking Oil,0.555556,0.625,1.125000
42,Rice,Biscuits,0.888889,1.000,1.125000
43,Biscuits,Rice,0.888889,1.000,1.125000
49,Sugar,Biscuits,0.222222,1.000,1.125000
...,...,...,...,...,...
3727,"Sugar, Tomato",Toothpaste,0.222222,1.000,1.125000
3728,Sugar,"Toothpaste, Tomato",0.222222,1.000,1.421053
3729,"Tea, Toothpaste",Tomato,0.370370,1.000,1.227273
3730,"Tea, Tomato",Toothpaste,0.370370,1.000,1.125000


The antecedents and consequents are converted from set notation into readable string format to improve clarity. This step enhances interpretability for non-technical stakeholders and allows the results to be directly used in reports and presentations. The final rule set highlights key co-purchase patterns such as Tea → Toothpaste & Tomato and Sugar → Toothpaste & Tomato, indicating unexpected but consistent purchasing associations.

# **Business Insights**

## 1. Staple Products Drive Basket Formation
Products such as Atta, Rice, Cooking Oil, Potato, and Soap show very high support, indicating that they are purchased in a majority of transactions. These items form the core of household shopping baskets and reflect habitual, low-discretion purchasing behavior.

**Business implication:**  
Staple products act as anchor items that drive store visits. Ensuring their availability and competitive pricing is critical, as they indirectly support the sale of complementary and higher-margin products.

---

## 2. Strong Complementary Purchase Patterns
Several association rules show high confidence and lift, indicating strong complementary relationships between products. Examples include Paneer with Tea and Milk, Sugar with Cooking Oil and Eggs, and Tea with Biscuits and Salt.

**Business implication:**  
Retailers can leverage these relationships for product bundling, combo offers, and strategic shelf placement to increase average basket value and improve shopping convenience.

---

## 3. Cross-Category Buying Behavior
The results reveal meaningful associations between grocery items and personal care products, such as Shampoo being associated with Toothpaste and Rice. This suggests that customers often combine essential grocery and personal care purchases in a single shopping trip.

**Business implication:**  
Cross-category promotions and adjacency-based shelf layouts can encourage impulse purchases and improve overall sales across departments.

---

## 4. Directional Nature of Association Rules
Many rules demonstrate directional behavior, where the purchase of one item strongly predicts the purchase of another, but not vice versa. This asymmetry highlights the importance of interpreting rules as directional insights rather than simple correlations.

**Business implication:**  
Directional rules are useful for recommendation systems, checkout prompts, and targeted promotions, where specific trigger products can be used to suggest complementary items.

---

## 5. Actionable Rule Filtering for Decision-Making
By applying thresholds on confidence and lift, the analysis focuses on non-random and managerially meaningful rules. This ensures that insights are practical and suitable for real-world retail applications rather than being purely statistical observations.

**Business implication:**  
Filtered rules provide a reliable basis for marketing and merchandising decisions without the risk of acting on coincidental associations.


# **CONCLUSION**

This market basket analysis demonstrates the effectiveness of association rule mining in uncovering meaningful purchasing patterns within Indian household consumption data. The results highlight the central role of staple products, strong complementary relationships across grocery and personal care categories, and significant opportunities for cross-selling and bundling strategies.

By applying the Apriori algorithm with carefully selected thresholds, the analysis balances statistical rigor with managerial interpretability. The findings can be directly applied to retail operations such as shelf optimization, promotional design, and customer recommendation systems. Overall, this study illustrates how data-driven insights can translate transactional data into actionable business intelligence, supporting more informed and strategic retail decision-making.